# 04 - Final Model and Kaggle Submission

## Objective

Train the selected SVR model using the complete labeled dataset
and generate price predictions for Kaggle's test dataset.

The preprocessing pipeline and selected hyperparameters will be
preserved from the previous experiments.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Localizar la raíz del proyecto.
possible_roots = [Path.cwd(), Path.cwd().parent]

ROOT_DIR = next(
    path for path in possible_roots
    if (path / "data" / "train.csv").exists()
    and (path / "src").exists()
)

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Cargar datos.
train = pd.read_csv(ROOT_DIR / "data" / "train.csv")
test = pd.read_csv(ROOT_DIR / "data" / "test.csv")
sample_submission = pd.read_csv(
    ROOT_DIR / "data" / "sample_submission.csv"
)

# Separar características y variable objetivo.
X = train.drop(columns=["SalePrice", "Id"])
y = train["SalePrice"].copy()

# Guardar los identificadores de test.
test_ids = test["Id"].copy()

# Eliminar identificador de las características.
X_test = test.drop(columns="Id")

# Comprobar compatibilidad.
assert set(X.columns) == set(X_test.columns)

# Alinear las columnas.
X_test = X_test[X.columns]

print("Entrenamiento:", X.shape)
print("Variable objetivo:", y.shape)
print("Test Kaggle:", X_test.shape)

Entrenamiento: (1460, 79)
Variable objetivo: (1460,)
Test Kaggle: (1459, 79)


In [2]:
from sklearn.svm import SVR
from sklearn.compose import TransformedTargetRegressor

from src.preprocessing import build_pipeline

# Definir SVR con los mejores hiperparámetros encontrados.
svr = SVR(
    kernel="rbf",
    C=5.342937261279771,
    epsilon=0.004678719265016202,
    gamma=0.001,
)

# Construir pipeline completo.
final_model = TransformedTargetRegressor(
    regressor=build_pipeline(
        X,
        svr,
        scale_numeric=True,
    ),
    func=np.log1p,
    inverse_func=np.expm1,
)

# Entrenar con las 1.460 viviendas etiquetadas.
final_model.fit(X, y)

print("Modelo final entrenado correctamente.")

Modelo final entrenado correctamente.


In [3]:
# Generar predicciones.
predictions = final_model.predict(X_test)

# Evitar predicciones negativas.
predictions = np.clip(predictions, 0, None)

print("Número de predicciones:", len(predictions))
print("Predicciones no finitas:", (~np.isfinite(predictions)).sum())

display(
    pd.Series(predictions, name="Predicted SalePrice").describe()
)

Número de predicciones: 1459
Predicciones no finitas: 0


count      1459.000000
mean     179253.898276
std       75859.836761
min       55747.048895
25%      128329.291498
50%      158643.510365
75%      208649.364335
max      531584.569396
Name: Predicted SalePrice, dtype: float64

In [4]:
# Comprobar que la plantilla y test contienen los mismos IDs.
assert set(test_ids) == set(sample_submission["Id"])

# Asociar cada predicción con su identificador.
prediction_by_id = pd.Series(
    predictions,
    index=test_ids,
)

# Mantener exactamente el orden de la plantilla oficial.
submission = sample_submission[["Id"]].copy()

submission["SalePrice"] = (
    submission["Id"].map(prediction_by_id)
)

# Validar la entrega.
assert len(submission) == len(test)
assert submission.columns.tolist() == (
    sample_submission.columns.tolist()
)
assert submission["Id"].is_unique
assert submission["SalePrice"].notna().all()
assert np.isfinite(submission["SalePrice"]).all()
assert (submission["SalePrice"] >= 0).all()

# Crear carpeta y guardar archivo.
output_dir = ROOT_DIR / "submissions"
output_dir.mkdir(exist_ok=True)

submission_path = output_dir / "submission.csv"

submission.to_csv(
    submission_path,
    index=False,
)

print("Archivo generado:", submission_path)
print("\nPrimeras predicciones:")
display(submission.head(10))

Archivo generado: c:\Users\afern\Desktop\Proyectos Personales\Kaggle - Houses Prices\submissions\submission.csv

Primeras predicciones:


,Id,SalePrice
0,1461,119441.570943
1,1462,164361.620022
2,1463,190021.661104
3,1464,201040.591043
4,1465,183967.375747
5,1466,172498.729519
6,1467,178345.658042
7,1468,164132.433221
8,1469,191910.330350
9,1470,125351.815578


## Final model and submission

The selected SVR model was retrained using all 1,460 labeled
observations from the original training dataset.

The model uses an RBF kernel and the hyperparameters selected
during cross-validation.

The preprocessing pipeline is included in the final model,
ensuring consistent transformations during training and prediction.

Predictions were generated for the 1,459 observations in Kaggle's
test dataset.

The resulting submission.csv was validated against the official
sample submission structure before being exported.